In [16]:
import pdfplumber
import pandas as pd
import re
import os

# STEP 1: Set base folder path
BASE_FOLDER = r"C:\Users\abidh\Documents"
OUTPUT_FOLDER = r"C:\Users\abidh\Documents"

FOLDERS = {
    "Monthly Gross Revenue Report 2022": ("casino_revenue", 2022),
    "Monthly Gross Revenue Report 2023": ("casino_revenue", 2023),
    "Monthly Gross Revenue Report 2024": ("casino_revenue", 2024),
    "Monthly Gross Revenue Report 2025": ("casino_revenue", 2025),
    "Monthly igaming tax return 2022":   ("igaming", 2022),
    "Monthly igaming tax return 2023":   ("igaming", 2023),
    "Monthly igaming tax return 2024":   ("igaming", 2024),
    "Monthly igaming tax return 2025":   ("igaming", 2025),
    "Monthly sports wagering tax return 2022": ("sports_wagering", 2022),
    "Monthly sports wagering tax return 2023": ("sports_wagering", 2023),
    "Monthly sports wagering tax return 2024": ("sports_wagering", 2024),
    "Monthly sports wagering tax return 2025": ("sports_wagering", 2025),
}

# STEP 2: Number extractor — handles split numbers like "$ 1 3,437,426"
def extract_first_dollar(line):
    """Extracts the first dollar amount from a line, handles split formatting."""
    # Match $ followed by digits/spaces/commas — joins split numbers like "$ 1 3,437,426"
    parts = re.findall(r'\$\s*([\d\s,]+)', line)
    if parts:
        combined = re.sub(r'[\s,]', '', parts[0])
        try:
            val = float(combined)
            return val if val > 0 else None
        except:
            return None
    # Fallback: match plain numbers (no $ sign) for sports wagering lines
    match = re.search(r'\b(\d{1,3}(?:,\d{3})+|\d{5,})\b', line)
    if match:
        try:
            return float(match.group().replace(',', ''))
        except:
            return None
    return None

def extract_last_dollar(line):
    """Extracts the LAST dollar amount from a line — useful for YTD fields."""
    parts = re.findall(r'\$\s*([\d\s,]+)', line)
    if parts:
        combined = re.sub(r'[\s,]', '', parts[-1])
        try:
            val = float(combined)
            return val if val > 0 else None
        except:
            return None
    return None

def clean_number(text):
    """Clean a raw number string including parentheses for negatives."""
    if not text:
        return None
    cleaned = re.sub(r'[\$,\s]', '', str(text))
    cleaned = cleaned.replace('(', '-').replace(')', '')
    try:
        return float(cleaned)
    except:
        return None

# STEP 3: Get month/year from text
def get_month_year(text, default_year):
    match = re.search(r'FOR THE MONTH OF\s+([A-Z]+)[,.\s]+(\d{4})', text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize(), int(match.group(2))
    return None, default_year

# STEP 4: Get casino name
def get_casino_name(text):
    match = re.search(r'^(.+?)\n(?:MONTHLY|FOR THE MONTH)', text, re.MULTILINE)
    if match:
        name = match.group(1).strip()
        name = re.sub(r',?\s*(LLC|LP|INC|CORP|LTD).*$', '', name, flags=re.IGNORECASE)
        name = re.sub(r'\s+dba\s*$', '', name, flags=re.IGNORECASE)
        if len(name) > 3:
            return name.strip()
    match = re.search(
        r'\n([A-Z][A-Z\s&\'.]+(?:CASINO|RESORT|HOTEL|GAMING|RACEWAY|RACETRACK|PARK)[A-Z\s&\'.]*)\n',
        text
    )
    if match:
        return match.group(1).strip()
    return "Unknown"

# STEP 5: Extract CASINO REVENUE
def extract_casino_revenue(lines, text, filename, year):
    data = {
        'data_type': 'casino_revenue',
        'source_file': filename,
        'casino': get_casino_name(text),
        'table_games_win': None,
        'poker_win': None,
        'slot_win': None,
        'total_casino_win': None,
        'ytd_gross_revenue': None,
    }
    data['month'], data['year'] = get_month_year(text, year)

    for line in lines:
        lu = line.upper()
        if re.match(r'^1\s+TABLE AND OTHER GAMES', lu):
            data['table_games_win'] = extract_first_dollar(line)
        elif re.match(r'^2\s+POKER', lu):
            data['poker_win'] = extract_first_dollar(line)
        elif re.match(r'^[34]\s+SLOT MACHINE', lu):
            data['slot_win'] = extract_first_dollar(line)
        elif re.match(r'^[345]\s+TOTAL CASINO WIN', lu):
            data['total_casino_win'] = extract_first_dollar(line)
        elif re.match(r'^10\s+YEAR-TO-DATE GROSS REVENUE', lu):
            data['ytd_gross_revenue'] = extract_last_dollar(line)
        elif re.match(r'^[789]\s+YEAR-TO-DATE', lu) and 'GROSS REVENUE' in lu:
            val = extract_last_dollar(line)
            if val:
                data['ytd_gross_revenue'] = val

    return data

# STEP 6: Extract IGAMING
def extract_igaming(lines, text, filename, year):
    if 'SKIN DETAIL' in text.upper():
        return None

    data = {
        'data_type': 'igaming',
        'source_file': filename,
        'casino': get_casino_name(text),
        'monthly_total_igaming_win': None,
        'ytd_total_igaming_win': None,
        'ytd_igaming_gross_revenue': None,
    }
    data['month'], data['year'] = get_month_year(text, year)

    for line in lines:
        lu = line.upper()
        if re.match(r'^3\s+TOTAL', lu):
            data['monthly_total_igaming_win'] = extract_first_dollar(line)
        elif re.match(r'^6\s+YEAR-TO-DATE TOTAL', lu):
            data['ytd_total_igaming_win'] = extract_last_dollar(line)
        elif re.match(r'^8\s+YEAR-TO-DATE INTERNET GAMING GROSS', lu):
            data['ytd_igaming_gross_revenue'] = extract_last_dollar(line)

    return data

# STEP 7: Extract SPORTS WAGERING
def extract_sports_wagering(lines, text, filename, year):
    data = {
        'data_type': 'sports_wagering',
        'source_file': filename,
        'casino': get_casino_name(text),
        'monthly_retail_sports_win': None,
        'monthly_internet_sports_win': None,
        'total_sports_tax_payment': None,
    }
    data['month'], data['year'] = get_month_year(text, year)

    for line in lines:
        lu = line.upper()
        if re.match(r'^3\s+MONTHLY RETAIL SPORTS', lu):
            val = extract_first_dollar(line)
            if val is None:
                match = re.search(r'[\d,]+$', line.strip())
                if match:
                    val = clean_number(match.group())
            data['monthly_retail_sports_win'] = val

        elif re.match(r'^16\s+MONTHLY (INTERNET|ONLINE) SPORTS', lu):
            val = extract_first_dollar(line)
            if val is None:
                match = re.search(r'[\d,]+$', line.strip())
                if match:
                    val = clean_number(match.group())
            data['monthly_internet_sports_win'] = val

        elif re.match(r'^26\s+TOTAL SPORTS WAGERING TAX', lu):
            data['total_sports_tax_payment'] = extract_last_dollar(line)

    return data

# STEP 8: Route each page to the correct extractor
def extract_page(page, filename, data_type, year):
    text = page.extract_text()
    if not text:
        return None

    lines = text.split('\n')
    tu = text.upper()

    if data_type == 'casino_revenue':
        if 'MONTHLY GROSS REVENUE REPORT' not in tu:
            return None
        return extract_casino_revenue(lines, text, filename, year)

    elif data_type == 'igaming':
        if 'MONTHLY INTERNET' not in tu:
            return None
        if 'SKIN DETAIL' in tu:
            return None
        return extract_igaming(lines, text, filename, year)

    elif data_type == 'sports_wagering':
        if 'MONTHLY SPORTS WAGERING TAX RETURN' not in tu:
            return None
        return extract_sports_wagering(lines, text, filename, year)

    return None

# STEP 9: Loop through all 12 folders
all_records = []
total_files = 0
total_pages = 0

for folder_name, (data_type, year) in FOLDERS.items():
    folder_path = os.path.join(BASE_FOLDER, folder_name)

    if not os.path.exists(folder_path):
        print(f"WARNING: Folder not found — {folder_path}")
        continue

    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')]
    print(f"📁 {folder_name} — {len(pdf_files)} PDFs found")
    total_files += len(pdf_files)

    for filename in pdf_files:
        filepath = os.path.join(folder_path, filename)
        try:
            with pdfplumber.open(filepath) as pdf:
                for page in pdf.pages:
                    record = extract_page(page, filename, data_type, year)
                    if record and record.get('casino') not in ["Unknown", None]:
                        all_records.append(record)
                        total_pages += 1
        except Exception as e:
            print(f"  ERROR: {filename} — {e}")

# STEP 10: Save to CSVs
df = pd.DataFrame(all_records)

df['date'] = pd.to_datetime(
    df['month'] + ' ' + df['year'].astype(str),
    format='%B %Y', errors='coerce'
)
df = df.sort_values(['data_type', 'casino', 'date']).reset_index(drop=True)

# Master CSV
master_path = os.path.join(OUTPUT_FOLDER, "all_casino_data.csv")
df.to_csv(master_path, index=False)

# Separate CSVs
for dtype in df['data_type'].unique():
    subset = df[df['data_type'] == dtype]
    out_path = os.path.join(OUTPUT_FOLDER, f"{dtype}_clean.csv")
    subset.to_csv(out_path, index=False)
    print(f"Saved {len(subset)} records → {dtype}_clean.csv")

print(f"\nDone!")
print(f"Total PDFs processed   : {total_files}")
print(f"Total records extracted: {total_pages}")
print(f"\nBreakdown:")
print(df['data_type'].value_counts())

# Preview Ocean Casino across all 3 data types
print("\nPreview — Casino Revenue (Ocean Casino):")
casino_df = df[df['data_type'] == 'casino_revenue']
ocean = casino_df[casino_df['casino'].str.contains('OCEAN', case=False, na=False)]
print(ocean[['casino','month','year','table_games_win','slot_win','total_casino_win']].head(10).to_string())

print("\nPreview — iGaming (Ocean Casino):")
ig_df = df[df['data_type'] == 'igaming']
ocean_ig = ig_df[ig_df['casino'].str.contains('OCEAN', case=False, na=False)]
print(ocean_ig[['casino','month','year','monthly_total_igaming_win','ytd_igaming_gross_revenue']].head(10).to_string())

print("\nPreview — Sports Wagering (Ocean Casino):")
sw_df = df[df['data_type'] == 'sports_wagering']
ocean_sw = sw_df[sw_df['casino'].str.contains('OCEAN', case=False, na=False)]
print(ocean_sw[['casino','month','year','monthly_retail_sports_win','monthly_internet_sports_win']].head(10).to_string())

📁 Monthly Gross Revenue Report 2022 — 12 PDFs found
📁 Monthly Gross Revenue Report 2023 — 12 PDFs found
📁 Monthly Gross Revenue Report 2024 — 12 PDFs found
📁 Monthly Gross Revenue Report 2025 — 12 PDFs found
📁 Monthly igaming tax return 2022 — 12 PDFs found
📁 Monthly igaming tax return 2023 — 12 PDFs found
📁 Monthly igaming tax return 2024 — 12 PDFs found
📁 Monthly igaming tax return 2025 — 12 PDFs found
📁 Monthly sports wagering tax return 2022 — 12 PDFs found
📁 Monthly sports wagering tax return 2023 — 12 PDFs found
📁 Monthly sports wagering tax return 2024 — 12 PDFs found
📁 Monthly sports wagering tax return 2025 — 12 PDFs found
Saved 432 records → casino_revenue_clean.csv
Saved 394 records → igaming_clean.csv
Saved 708 records → sports_wagering_clean.csv

Done!
Total PDFs processed   : 144
Total records extracted: 1534

Breakdown:
data_type
sports_wagering    708
casino_revenue     432
igaming            394
Name: count, dtype: int64

Preview — Casino Revenue (Ocean Casino):
      

In [17]:
import pandas as pd
df = pd.read_csv(r"C:\Users\abidh\Documents\casino_revenue_clean.csv")
print(df['casino'].unique())

["BALLY'S ATLANTIC CITY" 'BORGATA HOTEL CASINO & SPA'
 'CAESARS ATLANTIC CITY' 'GOLDEN NUGGET' 'HARD ROCK ATLANTIC CITY'
 "HARRAH'S ATLANTIC CITY" "HARRAH'S REPORT ATLANTIC CITY"
 "HARRAH'S RESORT ATLANTIC CITY" 'OCEAN CASINO RESORT'
 'OCEAN CASINO RESORT AC' 'PREMIER ENT AC' 'PREMIER ENTERTAINMENT AC'
 'RESORTS CASINO HOTEL' 'RESORTS CASINO HOTEL (DGMB CASINO'
 'TROPICANA CASINO & RESORT' 'TROPICANA CASINO & RESORTS']


In [19]:
import pandas as pd

name_map = {
    'PREMIER ENTERTAINMENT AC': "BALLY'S ATLANTIC CITY",
    'PREMIER ENT AC': "BALLY'S ATLANTIC CITY",
    'OCEAN CASINO RESORT AC': 'OCEAN CASINO RESORT',
    "HARRAH'S RESORT ATLANTIC CITY": "HARRAH'S ATLANTIC CITY",
    "HARRAH'S REPORT ATLANTIC CITY": "HARRAH'S ATLANTIC CITY",
    'RESORTS CASINO HOTEL (DGMB CASINO': 'RESORTS CASINO HOTEL',
    'TROPICANA CASINO & RESORTS': 'TROPICANA CASINO & RESORT',
}

for filename in ['casino_revenue_clean', 'igaming_clean', 'sports_wagering_clean', 'all_casino_data']:
    path = rf"C:\Users\abidh\Documents\{filename}.csv"
    df = pd.read_csv(path)
    df['casino'] = df['casino'].replace(name_map)
    df.to_csv(path, index=False)
    print(f"Fixed: {filename}.csv")

df = pd.read_csv(r"C:\Users\abidh\Documents\casino_revenue_clean.csv")
ballys = df[df['casino'] == "BALLY'S ATLANTIC CITY"]
print(f"\nBally's records: {len(ballys)}")
print(ballys['year'].value_counts().sort_index())

Fixed: casino_revenue_clean.csv
Fixed: igaming_clean.csv
Fixed: sports_wagering_clean.csv
Fixed: all_casino_data.csv

Bally's records: 48
year
2022    12
2023    12
2024    12
2025    12
Name: count, dtype: int64


In [21]:
import pandas as pd
df = pd.read_csv(r"C:\Users\abidh\Documents\casino_revenue_clean.csv")
print(df.groupby('casino')['year'].value_counts().unstack().fillna(0).astype(int))

year                        2022  2023  2024  2025
casino                                            
BALLY'S ATLANTIC CITY         12    12    12    12
BORGATA HOTEL CASINO & SPA    12    12    12    12
CAESARS ATLANTIC CITY         12    12    12    12
GOLDEN NUGGET                 12    12    12    12
HARD ROCK ATLANTIC CITY       12    12    12    12
HARRAH'S ATLANTIC CITY        12    12    12    12
OCEAN CASINO RESORT           12    12    12    12
RESORTS CASINO HOTEL          12    12    12    12
TROPICANA CASINO & RESORT     12    12    12    12


In [22]:
import pandas as pd
df = pd.read_csv(r"C:\Users\abidh\Documents\casino_revenue_clean.csv")
ballys = df[df['casino'].str.contains('BALLY', case=False, na=False)]
print(ballys[['casino', 'month', 'year', 'total_casino_win']].to_string())

                    casino      month  year  total_casino_win
0    BALLY'S ATLANTIC CITY    January  2025           9668742
1    BALLY'S ATLANTIC CITY   February  2025           9224853
2    BALLY'S ATLANTIC CITY      March  2025          11041065
3    BALLY'S ATLANTIC CITY      April  2025          10273137
4    BALLY'S ATLANTIC CITY        May  2025          11865007
5    BALLY'S ATLANTIC CITY       June  2025          11089646
6    BALLY'S ATLANTIC CITY       July  2025          14119163
7    BALLY'S ATLANTIC CITY     August  2025          14451730
8    BALLY'S ATLANTIC CITY  September  2025          10127933
9    BALLY'S ATLANTIC CITY    October  2025          11334316
10   BALLY'S ATLANTIC CITY   November  2025          10679285
11   BALLY'S ATLANTIC CITY   December  2025           9150715
300  BALLY'S ATLANTIC CITY    January  2023          11544104
301  BALLY'S ATLANTIC CITY   February  2023          12269167
302  BALLY'S ATLANTIC CITY      March  2023          10550249
303  BAL

In [23]:
import pandas as pd

for filename in ['casino_revenue_clean', 'igaming_clean', 'sports_wagering_clean', 'all_casino_data']:
    path = rf"C:\Users\abidh\Documents\{filename}.csv"
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['month'] + ' ' + df['year'].astype(str), format='%B %Y', errors='coerce')
    df = df.sort_values(['casino', 'date']).reset_index(drop=True)
    df.to_csv(path, index=False)
    print(f"Sorted: {filename}.csv")

print("Done! All files now sorted by casino then date.")

Sorted: casino_revenue_clean.csv
Sorted: igaming_clean.csv
Sorted: sports_wagering_clean.csv
Sorted: all_casino_data.csv
Done! All files now sorted by casino then date.


In [25]:
import pandas as pd

igaming_name_map = {
    'Premier Entertainment AC': "BALLY'S ATLANTIC CITY",
    'RESORTS DIGITAL GAMING': 'RESORTS CASINO HOTEL',
    'TROPICANA CASINO AND RESORT': 'TROPICANA CASINO & RESORT',
    'TROPICANA CASINO AND RESORT (ONLINE)': 'TROPICANA CASINO & RESORT',
    'CAESARS INTERACTIVE ENTERTAINMENT': 'CAESARS ATLANTIC CITY',
    '(AFFILIATE OF HARRAH\'S ATLANTIC CITY)': "HARRAH'S ATLANTIC CITY",
    '(AFFILIATE OF TROPICANA CASINO AND RESORT)': 'TROPICANA CASINO & RESORT',
    'Golden Nugget Online Gaming': 'GOLDEN NUGGET',
    'GOLDEN NUGGET ATLANTIC CITY': 'GOLDEN NUGGET',
}

for filename in ['igaming_clean', 'all_casino_data']:
    path = rf"C:\Users\abidh\Documents\{filename}.csv"
    df = pd.read_csv(path)
    df['casino'] = df['casino'].replace(igaming_name_map)
    df.to_csv(path, index=False)
    print(f"Fixed: {filename}.csv")

# Verify
df = pd.read_csv(r"C:\Users\abidh\Documents\igaming_clean.csv")
print("\nRecords per casino per year after fix:")
igaming_only = df[df['data_type'] == 'igaming']
print(igaming_only.groupby('casino')['year'].value_counts().unstack().fillna(0).astype(int))

Fixed: igaming_clean.csv
Fixed: all_casino_data.csv

Records per casino per year after fix:
year                        2022  2023  2024  2025
casino                                            
BALLY'S ATLANTIC CITY         12    12    12    12
BORGATA HOTEL CASINO & SPA    12    12    12    12
CAESARS ATLANTIC CITY         12    12     6     0
GOLDEN NUGGET                 13    12    12    12
HARD ROCK ATLANTIC CITY       12    12    12    12
HARRAH'S ATLANTIC CITY         0     3    12    12
OCEAN CASINO RESORT           12    12    12    12
RESORTS CASINO HOTEL          12    12    12    12
TROPICANA CASINO & RESORT     12    12    12    12


In [26]:
import pandas as pd

path = r"C:\Users\abidh\Documents\igaming_clean.csv"
df = pd.read_csv(path)

# Check the duplicate
gn_2022 = df[(df['casino'] == 'GOLDEN NUGGET') & (df['year'] == 2022)]
print(gn_2022[['casino', 'month', 'year', 'monthly_total_igaming_win']])

            casino      month  year  monthly_total_igaming_win
149  GOLDEN NUGGET       June  2022                 35705464.0
150  GOLDEN NUGGET       July  2022                 34529509.0
151  GOLDEN NUGGET     August  2022                 31376905.0
152  GOLDEN NUGGET  September  2022                 34167048.0
153  GOLDEN NUGGET    October  2022                 38383650.0
154  GOLDEN NUGGET   November  2022                 37710941.0
155  GOLDEN NUGGET   December  2022                 37786899.0
192  GOLDEN NUGGET        NaN  2022                 29693191.0
193  GOLDEN NUGGET    January  2022                 36485877.0
194  GOLDEN NUGGET   February  2022                 34560269.0
195  GOLDEN NUGGET      March  2022                 38280684.0
196  GOLDEN NUGGET      April  2022                 38142387.0
197  GOLDEN NUGGET        NaN  2022                  4910572.0


In [27]:
import pandas as pd

path = r"C:\Users\abidh\Documents\igaming_clean.csv"
df = pd.read_csv(path)

# Remove rows with missing month
before = len(df)
df = df.dropna(subset=['month'])
after = len(df)
print(f"Removed {before - after} rows with missing months")

df.to_csv(path, index=False)

# Verify Golden Nugget now has 12 records in 2022
gn = df[(df['casino'] == 'GOLDEN NUGGET') & (df['year'] == 2022)]
print(f"Golden Nugget 2022 records: {len(gn)}")
print(df.groupby('casino')['year'].value_counts().unstack().fillna(0).astype(int))

Removed 2 rows with missing months
Golden Nugget 2022 records: 11
year                        2022  2023  2024  2025
casino                                            
BALLY'S ATLANTIC CITY         12    12    12    12
BORGATA HOTEL CASINO & SPA    12    12    12    12
CAESARS ATLANTIC CITY         12    12     6     0
GOLDEN NUGGET                 11    12    12    12
HARD ROCK ATLANTIC CITY       12    12    12    12
HARRAH'S ATLANTIC CITY         0     3    12    12
OCEAN CASINO RESORT           12    12    12    12
RESORTS CASINO HOTEL          12    12    12    12
TROPICANA CASINO & RESORT     12    12    12    12


In [28]:
import pandas as pd

df = pd.read_csv(r"C:\Users\abidh\Documents\igaming_clean.csv")

gn_2022 = df[(df['casino'] == 'GOLDEN NUGGET') & (df['year'] == 2022)]
print(gn_2022[['casino', 'month', 'year', 'monthly_total_igaming_win']].to_string())

            casino      month  year  monthly_total_igaming_win
149  GOLDEN NUGGET       June  2022                 35705464.0
150  GOLDEN NUGGET       July  2022                 34529509.0
151  GOLDEN NUGGET     August  2022                 31376905.0
152  GOLDEN NUGGET  September  2022                 34167048.0
153  GOLDEN NUGGET    October  2022                 38383650.0
154  GOLDEN NUGGET   November  2022                 37710941.0
155  GOLDEN NUGGET   December  2022                 37786899.0
192  GOLDEN NUGGET    January  2022                 36485877.0
193  GOLDEN NUGGET   February  2022                 34560269.0
194  GOLDEN NUGGET      March  2022                 38280684.0
195  GOLDEN NUGGET      April  2022                 38142387.0


In [29]:
import pandas as pd

df = pd.read_csv(r"C:\Users\abidh\Documents\igaming_clean.csv")

print(df.groupby(['casino', 'year'])['month'].count().unstack().fillna(0).astype(int))

year                        2022  2023  2024  2025
casino                                            
BALLY'S ATLANTIC CITY         12    12    12    12
BORGATA HOTEL CASINO & SPA    12    12    12    12
CAESARS ATLANTIC CITY         12    12     6     0
GOLDEN NUGGET                 11    12    12    12
HARD ROCK ATLANTIC CITY       12    12    12    12
HARRAH'S ATLANTIC CITY         0     3    12    12
OCEAN CASINO RESORT           12    12    12    12
RESORTS CASINO HOTEL          12    12    12    12
TROPICANA CASINO & RESORT     12    12    12    12


In [30]:
import pandas as pd

df = pd.read_csv(r"C:\Users\abidh\Documents\casino_revenue_clean.csv")
casino_df = df[['casino', 'month', 'year', 'table_games_win', 'slot_win', 'total_casino_win']]
casino_df.to_csv(r"C:\Users\abidh\Documents\casino_revenue_final.csv", index=False)
print("Casino Revenue:")
print(casino_df.head())

Casino Revenue:
                  casino     month  year  table_games_win  slot_win  \
0  BALLY'S ATLANTIC CITY   January  2022          3045482   5738134   
1  BALLY'S ATLANTIC CITY  February  2022          3115899   7620862   
2  BALLY'S ATLANTIC CITY     March  2022          5783567   7694689   
3  BALLY'S ATLANTIC CITY     April  2022          5003408   9415408   
4  BALLY'S ATLANTIC CITY       May  2022          2351636  10319057   

   total_casino_win  
0           8783616  
1          10736761  
2          13478256  
3          14418816  
4          12670693  


In [31]:
df = pd.read_csv(r"C:\Users\abidh\Documents\igaming_clean.csv")
igaming_df = df[['casino', 'month', 'year', 'monthly_total_igaming_win']]
igaming_df.to_csv(r"C:\Users\abidh\Documents\igaming_final.csv", index=False)
print("\niGaming:")
print(igaming_df.head())


iGaming:
                   casino      month  year  monthly_total_igaming_win
0  HARRAH'S ATLANTIC CITY  September  2024                  1574804.0
1  HARRAH'S ATLANTIC CITY    October  2024                  1848249.0
2  HARRAH'S ATLANTIC CITY   November  2024                  1377376.0
3  HARRAH'S ATLANTIC CITY   December  2024                  1444650.0
4  HARRAH'S ATLANTIC CITY    January  2025                  1276401.0


In [32]:
df = pd.read_csv(r"C:\Users\abidh\Documents\sports_wagering_clean.csv")
sports_df = df[['casino', 'month', 'year', 'monthly_retail_sports_win', 'monthly_internet_sports_win']]
sports_df['total_sports_win'] = sports_df['monthly_retail_sports_win'].fillna(0) + sports_df['monthly_internet_sports_win'].fillna(0)
sports_df.to_csv(r"C:\Users\abidh\Documents\sports_wagering_final.csv", index=False)
print("\nSports Wagering:")
print(sports_df.head())


Sports Wagering:
                                  casino      month  year  \
0  (AFFILIATE OF HARRAH'S ATLANTIC CITY)  September  2024   
1  (AFFILIATE OF HARRAH'S ATLANTIC CITY)    October  2024   
2  (AFFILIATE OF HARRAH'S ATLANTIC CITY)   November  2024   
3  (AFFILIATE OF HARRAH'S ATLANTIC CITY)   December  2024   
4  (AFFILIATE OF HARRAH'S ATLANTIC CITY)    January  2025   

   monthly_retail_sports_win  monthly_internet_sports_win  total_sports_win  
0                        NaN                      54266.0           54266.0  
1                        NaN                     128692.0          128692.0  
2                        NaN                     115995.0          115995.0  
3                        NaN                     171537.0          171537.0  
4                        NaN                     297320.0          297320.0  


C:\Users\abidh\AppData\Local\Temp\ipykernel_30140\180203620.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sports_df['total_sports_win'] = sports_df['monthly_retail_sports_win'].fillna(0) + sports_df['monthly_internet_sports_win'].fillna(0)


In [33]:
import pandas as pd

# CASINO REVENUE 
df = pd.read_csv(r"C:\Users\abidh\Documents\casino_revenue_clean.csv")
casino_df = df[['casino', 'month', 'year', 'table_games_win', 'slot_win', 'total_casino_win']].copy()
casino_df.to_csv(r"C:\Users\abidh\Documents\casino_revenue_final.csv", index=False)
print(f"Casino Revenue: {len(casino_df)} records")
print(casino_df['casino'].unique())

# IGAMING 
df = pd.read_csv(r"C:\Users\abidh\Documents\igaming_clean.csv")
igaming_df = df[['casino', 'month', 'year', 'monthly_total_igaming_win']].copy()

# Remove affiliates and non-casino entities
exclude = ['AFFILIATE', 'INTERACTIVE', 'DIGITAL', 'ONLINE GAMING']
mask = ~igaming_df['casino'].str.upper().str.contains('|'.join(exclude), na=False)
igaming_df = igaming_df[mask]
igaming_df.to_csv(r"C:\Users\abidh\Documents\igaming_final.csv", index=False)
print(f"\niGaming: {len(igaming_df)} records")
print(igaming_df['casino'].unique())

# SPORTS WAGERING
df = pd.read_csv(r"C:\Users\abidh\Documents\sports_wagering_clean.csv")
sports_df = df[['casino', 'month', 'year', 'monthly_retail_sports_win', 'monthly_internet_sports_win']].copy()

# Remove affiliates and racetracks
exclude_sw = ['AFFILIATE', 'MEADOWLANDS', 'MONMOUTH', 'FREEHOLD', 'DARBY', 
              'FR PARK', 'BOARDWALK REGENCY', 'DIGITAL', 'OPERATING', 'ONLINE']
mask = ~sports_df['casino'].str.upper().str.contains('|'.join(exclude_sw), na=False)
sports_df = sports_df[mask]

# Standardize names
sw_name_map = {
    'HARD ROCK HOTEL & CASINO ATLANTIC CITY': 'HARD ROCK ATLANTIC CITY',
    'HARD ROCK HOTEL AND CASINO ATLANTIC CITY': 'HARD ROCK ATLANTIC CITY',
    'Borgata Hotel Casino & Spa': 'BORGATA HOTEL CASINO & SPA',
    'TROPICANA CASINO AND RESORT': 'TROPICANA CASINO & RESORT',
    'Tropicana Casino & Resort': 'TROPICANA CASINO & RESORT',
    'Tropicana Casino & Resort (Online)': 'TROPICANA CASINO & RESORT',
    'GOLDEN NUGGET ONLINE GAMING': 'GOLDEN NUGGET',
    "HARRAH'S ATLANTIC CITY OPERATING": "HARRAH'S ATLANTIC CITY",
    'Darby Development (Monmouth Park)': None,
}
sports_df['casino'] = sports_df['casino'].replace(sw_name_map)

# Create total sports win column
sports_df['total_sports_win'] = sports_df['monthly_retail_sports_win'].fillna(0) + sports_df['monthly_internet_sports_win'].fillna(0)
sports_df.to_csv(r"C:\Users\abidh\Documents\sports_wagering_final.csv", index=False)
print(f"\nSports Wagering: {len(sports_df)} records")
print(sports_df['casino'].unique())

Casino Revenue: 432 records
["BALLY'S ATLANTIC CITY" 'BORGATA HOTEL CASINO & SPA'
 'CAESARS ATLANTIC CITY' 'GOLDEN NUGGET' 'HARD ROCK ATLANTIC CITY'
 "HARRAH'S ATLANTIC CITY" 'OCEAN CASINO RESORT' 'RESORTS CASINO HOTEL'
 'TROPICANA CASINO & RESORT']

iGaming: 392 records
["HARRAH'S ATLANTIC CITY" 'TROPICANA CASINO & RESORT'
 "BALLY'S ATLANTIC CITY" 'BORGATA HOTEL CASINO & SPA'
 'CAESARS ATLANTIC CITY' 'GOLDEN NUGGET' 'HARD ROCK ATLANTIC CITY'
 'OCEAN CASINO RESORT' 'RESORTS CASINO HOTEL']

Sports Wagering: 456 records
["BALLY'S ATLANTIC CITY" 'BORGATA HOTEL CASINO & SPA'
 'CAESARS ATLANTIC CITY' 'GOLDEN NUGGET' 'HARD ROCK ATLANTIC CITY'
 "HARRAH'S ATLANTIC CITY" 'OCEAN CASINO RESORT' 'RESORTS CASINO HOTEL'
 'TROPICANA CASINO & RESORT']


In [34]:
import pandas as pd

print("Casino Revenue Final columns:")
df = pd.read_csv(r"C:\Users\abidh\Documents\casino_revenue_final.csv")
print(df.columns.tolist())

print("\niGaming Final columns:")
df = pd.read_csv(r"C:\Users\abidh\Documents\igaming_final.csv")
print(df.columns.tolist())

print("\nSports Wagering Final columns:")
df = pd.read_csv(r"C:\Users\abidh\Documents\sports_wagering_final.csv")
print(df.columns.tolist())

Casino Revenue Final columns:
['casino', 'month', 'year', 'table_games_win', 'slot_win', 'total_casino_win']

iGaming Final columns:
['casino', 'month', 'year', 'monthly_total_igaming_win']

Sports Wagering Final columns:
['casino', 'month', 'year', 'monthly_retail_sports_win', 'monthly_internet_sports_win', 'total_sports_win']
